# ⚡ Module 10: Ensemble Methods — Boosting
## Theory + Practical

---

## 📚 THEORY

### Bagging vs Boosting

| | Bagging (Random Forest) | Boosting (XGBoost, AdaBoost) |
|-|------------------------|-----------------------------|
| Trees | Parallel (independent) | Sequential (each fixes previous) |
| Goal | Reduce variance | Reduce bias |
| Focus | Random samples | Hard misclassified samples |

### AdaBoost
```
1. Train weak learner (e.g., shallow tree) on data
2. Give higher weight to misclassified samples
3. Train next learner focusing on those hard samples
4. Repeat — final model = weighted sum of all learners
```

### Gradient Boosting / XGBoost
```
1. Start with a simple prediction (e.g., mean)
2. Compute residuals (errors of current model)
3. Train a tree to predict those residuals
4. Add that tree to the model: prediction += learning_rate × tree
5. Repeat until stopping criterion
```

**Key parameters:**
- `n_estimators` — number of trees
- `learning_rate` — how much each tree contributes (smaller = slower but better)
- `max_depth` — complexity of each tree (usually 3-6 for boosting)

---

## 🔬 PRACTICAL

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
import warnings; warnings.filterwarnings('ignore')

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Ready ✅')

### Practical 1: AdaBoost — Weak Learners Become Strong

In [ ]:
# A single stump (depth=1 tree) is a weak learner
stump = DecisionTreeClassifier(max_depth=1)
stump.fit(X_train, y_train)
stump_acc = accuracy_score(y_test, stump.predict(X_test))

# AdaBoost: combine 100 stumps
ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                          n_estimators=100, learning_rate=0.1, random_state=42)
ada.fit(X_train, y_train)
ada_acc = accuracy_score(y_test, ada.predict(X_test))

print(f'Single Stump (weak): {stump_acc:.4f}')
print(f'AdaBoost (100 stumps): {ada_acc:.4f}')
print(f'Improvement: +{(ada_acc - stump_acc)*100:.1f}%')

### Practical 2: Gradient Boosting

In [ ]:
# Track training and test accuracy as we add more trees
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                  max_depth=3, random_state=42)
gb.fit(X_train, y_train)

# staged_predict gives prediction after each tree
train_scores, test_scores = [], []
for y_pred in gb.staged_predict(X_train):
    train_scores.append(accuracy_score(y_train, y_pred))
for y_pred in gb.staged_predict(X_test):
    test_scores.append(accuracy_score(y_test, y_pred))

best_n = np.argmax(test_scores) + 1

plt.figure(figsize=(8, 4))
plt.plot(train_scores, label='Train', color='blue')
plt.plot(test_scores, label='Test', color='orange')
plt.axvline(best_n - 1, color='red', linestyle='--', label=f'Best n_estimators={best_n}')
plt.title('Gradient Boosting: Accuracy vs Number of Trees')
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Best at {best_n} trees: test accuracy = {test_scores[best_n-1]:.4f}')

### Practical 3: Compare All Ensemble Methods

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest (Bagging)': RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

for name, m in models.items():
    scores = cross_val_score(m, X, y, cv=5, scoring='accuracy')
    print(f'{name:<30} Accuracy: {scores.mean():.4f} ± {scores.std():.4f}')

### 🏋️ Try It Yourself!

In [ ]:
# ✏️ YOUR TURN: Tune learning_rate and n_estimators
# Small learning_rate + many trees = usually better

lr = 0.05          # Try: 0.01, 0.05, 0.1, 0.5
n_trees = 200      # Try: 50, 100, 200, 500

gb_exp = GradientBoostingClassifier(learning_rate=lr, n_estimators=n_trees,
                                     max_depth=3, random_state=42)
scores = cross_val_score(gb_exp, X, y, cv=5, scoring='accuracy')
print(f'lr={lr}, n_trees={n_trees}')
print(f'CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}')

## 🎓 Summary

| Method | Key Idea |
|--------|----------|
| Bagging (RF) | Parallel trees on random subsets → reduce variance |
| AdaBoost | Sequential; upweight hard samples |
| Gradient Boosting | Sequential; fit residuals with new trees |
| learning_rate | Smaller = slower but more precise |
| Boosting tip | Shallow trees (max_depth=3-5) work best |

➡️ **Next: Neural Networks Introduction**